# Deadline 1 — MelBERT (VUA-18) trên Kaggle
Paper: NAACL 2021 (aclanthology 2021.naacl-main.141) · Code chính thức: https://github.com/jin530/MelBERT

**Trước khi chạy:** Settings → Accelerator = **GPU (T4/P100)**, Internet = **On**. Chạy tuần tự từ trên xuống.

Quy trình:
1. Clone repo, ghi commit; cài thư viện; tải dữ liệu VUA-18 do tác giả cung cấp
2. Kiểm tra số liệu split so với README (train 116,622 / dev 38,628 / test 50,175 token)
3. Train **toàn bộ train**, **chọn checkpoint theo dev**, rồi đánh giá **toàn bộ test**
4. Xuất dự đoán từng token (CSV), tính P/R/F1 (tổng + VERB/NOUN/ADJ/ADV), lưu cấu hình/log/checkpoint/thời gian/số tham số/phần cứng/khác biệt

Đổi `MODEL_TYPE = "BERT_BASE"` ở ô cấu hình để chạy RoBERTa thuần cho Bạn 2 (cùng notebook).

## 0. Cấu hình

In [1]:
import os, time, json, shutil, platform, subprocess
MODEL_TYPE = "BERT_BASE"          # MELBERT | BERT_BASE (RoBERTa thuần) | MELBERT_SPV | MELBERT_MIP | BERT_SEQ
BERT_MODEL = "roberta-base"
DATA_GDRIVE_ID = "18hemekvUuOw-qkQhWv4F6qFnis6lkaog"   # link dữ liệu trong README của tác giả
PIN_TRANSFORMERS = ["4.48.3", "4.46.3"]   # thử lần lượt; repo gốc dùng 4.2.2 (py3.7) không cài được trên Kaggle py3.12/3.13.
                                           # KHÔNG dùng transformers 5.x: đổi tokenizer/API -> gây AssertionError segment_ids. Đặt [] để giữ bản có sẵn.
EXTRA_ARGS = {}                 # CHỈ điền khi cố ý đổi so với main_config.cfg, vd {"seed": 42}; mọi thay đổi phải ghi báo cáo

WORK = "/kaggle/working"
REPO = f"{WORK}/MelBERT"
OUT  = f"{WORK}/results_{MODEL_TYPE}"
os.makedirs(OUT, exist_ok=True)
DIFFS = []   # danh sách khác biệt so với tác giả, sẽ lưu vào run_info.json
TQDM = "TQDM_MININTERVAL=60"
print(MODEL_TYPE, BERT_MODEL, OUT)

BERT_BASE roberta-base /kaggle/working/results_BERT_BASE


## 1. Clone repo + cài môi trường

In [2]:
!git clone https://github.com/jin530/MelBERT.git {REPO}
COMMIT = subprocess.check_output(f"cd {REPO} && git rev-parse HEAD", shell=True, text=True).strip()
print("commit:", COMMIT)

Cloning into '/kaggle/working/MelBERT'...
remote: Enumerating objects: 98, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 98 (delta 37), reused 69 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (98/98), 400.46 KiB | 8.34 MiB/s, done.
Resolving deltas: 100% (37/37), done.
commit: 11b0f78dae93c71c56b805dc28d810f6050c86c4


In [3]:
!pip install -q gdown scikit-learn pandas
import importlib
if PIN_TRANSFORMERS:
    ok = None
    for v in PIN_TRANSFORMERS:
        rc = subprocess.run(f'pip install -q "transformers=={v}"', shell=True).returncode
        if rc == 0:
            ok = v; break
        print(f"không cài được transformers=={v}, thử bản kế tiếp")
    assert ok, "Không cài được bản transformers nào trong PIN_TRANSFORMERS (kiểm tra Internet = On)"
    DIFFS.append(f"transformers=={ok} thay vì 4.2.2 (python mới trên Kaggle, torch mới thay vì 1.6)")
    print("đã cài transformers", ok, "-> nhớ RESTART kernel nếu ô tiếp theo vẫn báo phiên bản khác")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


đã cài transformers 4.48.3 -> nhớ RESTART kernel nếu ô tiếp theo vẫn báo phiên bản khác


In [4]:
!nvidia-smi
import torch, transformers, sys
print(sys.version); print('torch', torch.__version__, 'transformers', transformers.__version__, 'cuda', torch.cuda.is_available())
assert not transformers.__version__.startswith("5"), (
    f"Đang là transformers {transformers.__version__} (5.x). Restart kernel (Run -> Restart session) rồi chạy lại từ ô 0 "
    "để bản đã pin có hiệu lực.")
assert torch.cuda.is_available(), (
    "Không thấy GPU (log trước đây in 'device: cpu'). Settings -> Accelerator -> GPU T4/P100, rồi restart session. "
    "Train trên CPU sẽ không xong.")

Mon Nov 16 23:24:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Tải dữ liệu VUA-18 (bản tác giả cung cấp)

In [5]:
import gdown, zipfile, tarfile, glob
RAW = f"{WORK}/raw"; os.makedirs(RAW, exist_ok=True)
f = gdown.download(id=DATA_GDRIVE_ID, output=f"{RAW}/data_download", quiet=False)
if zipfile.is_zipfile(f):
    zipfile.ZipFile(f).extractall(RAW)
elif tarfile.is_tarfile(f):
    tarfile.open(f).extractall(RAW)
for p in sorted(glob.glob(f"{RAW}/**/*.tsv", recursive=True)):
    print(os.path.relpath(p, RAW))

Downloading...
From: https://drive.google.com/uc?id=18hemekvUuOw-qkQhWv4F6qFnis6lkaog
To: /kaggle/working/raw/data_download
100%|██████████| 16.3M/16.3M [00:00<00:00, 24.6MB/s]


MOH-X/CLS/test0.tsv
MOH-X/CLS/test1.tsv
MOH-X/CLS/test2.tsv
MOH-X/CLS/test3.tsv
MOH-X/CLS/test4.tsv
MOH-X/CLS/test5.tsv
MOH-X/CLS/test6.tsv
MOH-X/CLS/test7.tsv
MOH-X/CLS/test8.tsv
MOH-X/CLS/test9.tsv
MOH-X/CLS/train0.tsv
MOH-X/CLS/train1.tsv
MOH-X/CLS/train2.tsv
MOH-X/CLS/train3.tsv
MOH-X/CLS/train4.tsv
MOH-X/CLS/train5.tsv
MOH-X/CLS/train6.tsv
MOH-X/CLS/train7.tsv
MOH-X/CLS/train8.tsv
MOH-X/CLS/train9.tsv
TroFi/CLS/test0.tsv
TroFi/CLS/test1.tsv
TroFi/CLS/test2.tsv
TroFi/CLS/test3.tsv
TroFi/CLS/test4.tsv
TroFi/CLS/test5.tsv
TroFi/CLS/test6.tsv
TroFi/CLS/test7.tsv
TroFi/CLS/test8.tsv
TroFi/CLS/test9.tsv
TroFi/CLS/train0.tsv
TroFi/CLS/train1.tsv
TroFi/CLS/train2.tsv
TroFi/CLS/train3.tsv
TroFi/CLS/train4.tsv
TroFi/CLS/train5.tsv
TroFi/CLS/train6.tsv
TroFi/CLS/train7.tsv
TroFi/CLS/train8.tsv
TroFi/CLS/train9.tsv
VUA18/dev.tsv
VUA18/test.tsv
VUA18/train.tsv
VUA18_genre/CLS/acad/test.tsv
VUA18_genre/CLS/conv/test.tsv
VUA18_genre/CLS/fict/test.tsv
VUA18_genre/CLS/news/test.tsv
VUA18_pos/adj/t

In [6]:
# Chọn thư mục VUA18 gốc (có train/dev/test.tsv, không phải tập con theo pos/genre)
cands = [os.path.dirname(p) for p in glob.glob(f"{RAW}/**/train.tsv", recursive=True)
         if "vua18" in p.lower() and "genre" not in p.lower() and "/pos" not in p.lower()]
print(cands)
DATA_DIR = cands[0]          # nếu sai, gán tay: DATA_DIR = ".../VUA18"
print("DATA_DIR =", DATA_DIR, os.listdir(DATA_DIR))

['/kaggle/working/raw/VUA18']
DATA_DIR = /kaggle/working/raw/VUA18 ['train.tsv', 'dev.tsv', 'test.tsv']


In [7]:
import pandas as pd, csv
def read_tsv(p): return pd.read_csv(p, sep="\t", quoting=csv.QUOTE_NONE, dtype=str, keep_default_na=False)
stats = {}
for s in ["train", "dev", "test"]:
    d = read_tsv(f"{DATA_DIR}/{s}.tsv")
    stats[s] = dict(tokens=len(d), pct_metaphor=round(100*(d.label=="1").mean(), 1), sentences=d.sentence.nunique())
print(pd.DataFrame(stats).T)
print("README:  train 116,622 / 11.2% / 6,323 | dev 38,628 / 11.6% / 1,550 | test 50,175 / 12.4% / 2,694")
print("POS trong test:", read_tsv(f"{DATA_DIR}/test.tsv").POS.value_counts().to_dict())

         tokens  pct_metaphor  sentences
train  101975.0          12.8     6268.0
dev     34253.0          13.1     1541.0
test    43947.0          14.2     2679.0
README:  train 116,622 / 11.2% / 6,323 | dev 38,628 / 11.6% / 1,550 | test 50,175 / 12.4% / 2,694
POS trong test: {'VERB': 9247, 'NOUN': 8583, 'ADP': 5300, 'DET': 4118, 'ADJ': 3965, 'PRON': 3955, 'ADV': 3054, 'PROPN': 1905, 'CCONJ': 1581, 'PART': 1257, 'NUM': 543, 'INTJ': 398, 'PUNCT': 33, 'X': 7, 'SYM': 1}


## 3. Kiểm tra code của tác giả — cách chọn checkpoint

`main.py` (chính thức) gọi `load_test_data(...)` sau **mỗi epoch** và lưu checkpoint khi F1 trên dữ liệu đó cao nhất. Ô dưới in ra file cấu hình và chỗ đọc `*.tsv` để xác nhận `load_test_data` đọc `test.tsv`.
Nếu đúng, chạy nguyên bản sẽ **chọn checkpoint theo test** — trái với quy ước nhóm (chọn theo dev). Cách xử lý không sửa code: tạo thư mục `VUA18_devsel` với `test.tsv := dev.tsv` để train/chọn checkpoint bằng dev, sau đó đánh giá trên test thật. Ghi vào báo cáo như một khác biệt có chủ đích.

In [8]:
!echo "===== main_config.cfg ====="; cat {REPO}/main_config.cfg
!echo; echo "===== chỗ đọc tsv ====="; grep -n "tsv\|get_.*_examples" {REPO}/data_loader.py {REPO}/run_classifier_dataset_utils.py | head -40
!echo; echo "===== utils ====="; ls {REPO}/utils

===== main_config.cfg =====
[args]
# Bert pre-trained model selected in the list [bert-base-cased, roberta-base, albert-base-v1 / albert-large-v1] (default = roberta-base)
bert_model = roberta-base

# The input data dir. Should contain the .tsv files (VUA18 / VUAverb / MOH-X/CLS / TroFi/CLS / VUA20)
data_dir = data/VUA20
# The name of the task to train (vua(1-fold) / trofi(10-fold))
task_name = vua
# The name of model type (default = MELBERT) (BERT_BASE / BERT_SEQ / MELBERT_SPV / MELBERT_MIP / MELBERT)
model_type = MELBERT
# The hidden dimension for classifier (default = 768)
classifier_hidden = 768
# Learning rate scheduler (default = warmup_linear) (none / warmup_linear)
lr_schedule = warmup_linear
# Training epochs to perform linear learning rate warmup for. (default = 2)
warmup_epoch = 2
# Dropout ratio (default = 0.2)
drop_ratio = 0.2
# K-fold (default = 10)
kfold = 10
# Number of bagging (default = 0) (0 not for using bagging technique)
num_bagging = 0
# The index of bagging only

In [9]:
# Đối chiếu cấu hình với paper (điền/kiểm tra bằng mắt): epochs=3, batch=32, lr=3e-5, max_seq_len=150, class_weight=3, warmup_epoch=2, drop_ratio=0.2
# Nếu main_config.cfg khác paper -> giữ NGUYÊN cfg và ghi chú; không tự đổi.
DEVSEL = f"{WORK}/data_devsel/VUA18"
os.makedirs(DEVSEL, exist_ok=True)
shutil.copy(f"{DATA_DIR}/train.tsv", f"{DEVSEL}/train.tsv")
shutil.copy(f"{DATA_DIR}/dev.tsv",   f"{DEVSEL}/dev.tsv")
shutil.copy(f"{DATA_DIR}/dev.tsv",   f"{DEVSEL}/test.tsv")     # "test" ảo = dev -> chọn checkpoint theo dev
DIFFS.append("Chọn checkpoint theo dev: thư mục train có test.tsv:=dev.tsv (code gốc chọn theo tập do load_test_data đọc). Test thật chỉ dùng sau khi train xong.")
DIFFS.append("Chạy 1 GPU (CUDA_VISIBLE_DEVICES=0), không DataParallel.")

In [10]:
# Smoke test import (phát hiện sớm lỗi tương thích transformers)
r = subprocess.run(f"cd {REPO} && python -c 'import main'", shell=True, capture_output=True, text=True)
print("OK" if r.returncode == 0 else r.stderr[-3000:])

OK


In [11]:
# CHỈ chạy ô này nếu ô trên báo ImportError về AdamW (bản transformers mới đã bỏ AdamW).
# Thay bằng torch.optim.AdamW với eps=1e-6 (mặc định của transformers.AdamW cũ).
if r.returncode != 0 and "AdamW" in r.stderr:
    s = open(f"{REPO}/main.py").read()
    s = s.replace("from transformers import AutoTokenizer, AutoModel, AdamW, get_linear_schedule_with_warmup",
                  "from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup\nfrom torch.optim import AdamW")
    s = s.replace("AdamW(optimizer_grouped_parameters, lr=args.learning_rate)",
                  "AdamW(optimizer_grouped_parameters, lr=args.learning_rate, eps=1e-6)")
    open(f"{REPO}/main.py", "w").write(s)
    DIFFS.append("main.py: dùng torch.optim.AdamW(eps=1e-6) thay transformers.AdamW (đã bị gỡ)")
    r = subprocess.run(f"cd {REPO} && python -c 'import main'", shell=True, capture_output=True, text=True)
    print("OK" if r.returncode == 0 else r.stderr[-3000:])

### 3a. Vá lỗi `segment_ids` của repo gốc
Trong `convert_two_examples_to_features`, `tokens` được nối thêm `text_b + [SEP]` (`len(text_b)+1` token) nhưng `segment_ids` chỉ nối `[0]*len(text_b)` -> luôn ngắn hơn 1 -> `AssertionError`. Ô dưới sửa thành `[0]*(len(text_b)+1)` (chỉ đổi độ dài đệm, không đổi vị trí token đích có segment=1).

In [12]:
p = f"{REPO}/run_classifier_dataset_utils.py"
s = open(p).read()
a = s.index("def convert_two_examples_to_features")
b = s.find("\ndef ", a + 10); b = len(s) if b == -1 else b
fn = s[a:b]
old = "segment_ids = segment_ids + [0] * len(text_b)"
new = "segment_ids = segment_ids + [0] * (len(text_b) + 1)  # +1 cho [SEP] cuối"
if old in fn:
    assert fn.count(old) == 1, f"tìm thấy {fn.count(old)} chỗ, kiểm tra tay"
    open(p, "w").write(s[:a] + fn.replace(old, new) + s[b:])
    DIFFS.append("run_classifier_dataset_utils.py: segment_ids đệm thêm 1 cho [SEP] cuối (lỗi độ dài trong code gốc, assert len(segment_ids)==max_seq_length luôn fail)")
    print("đã vá")
elif new in fn:
    print("đã vá từ trước")
else:
    raise RuntimeError("không thấy dòng cần vá; dán lại mã nguồn hàm cho mình")

đã vá


### 3b. Kiểm tra trước converter (tránh train chết sau 0.3 phút)
Ô dưới chạy hàm `convert_two_examples_to_features` của tác giả từng ví dụ một trên 5,000 dòng đầu của train, in ra mã nguồn quanh dòng `assert` và các câu làm lệch `segment_ids`. Nếu thấy `OK`, sang mục 4. Nếu có lỗi, copy toàn bộ output của ô này để sửa tiếp đúng chỗ.

In [13]:
import sys, inspect
sys.path.insert(0, REPO); os.chdir(REPO)
for m in [m for m in sys.modules if m in ("run_classifier_dataset_utils", "data_loader")]: del sys.modules[m]
import transformers
from transformers import AutoTokenizer
import run_classifier_dataset_utils as U
print("transformers", transformers.__version__)

_src = inspect.getsource(U.convert_two_examples_to_features).splitlines()
for i, l in enumerate(_src):
    if "assert" in l:
        print("\n".join(f"{j:4d}: {_src[j]}" for j in range(max(0, i-3), min(len(_src), i+2))))
        print("-----")

proc = U.processors["vua"](); labels = proc.get_labels()
tok = AutoTokenizer.from_pretrained(BERT_MODEL)
exs = proc.get_train_examples(DEVSEL)
print("số ví dụ:", len(exs), "| tokenizer:", type(tok).__name__)
bad = []
for ex in exs[:5000]:
    try:
        U.convert_two_examples_to_features([ex], labels, 256, tok, U.output_modes["vua"])
    except AssertionError:
        bad.append(ex)
print("OK, converter chạy sạch 5,000 ví dụ đầu" if not bad else f"LỖI: {len(bad)}/5000 ví dụ làm assert fail")
for ex in bad[:3]:
    words = ex.text_a.split()
    whole = tok.tokenize(ex.text_a)
    per_word = [len(tok.tokenize(w if i == 0 else " " + w)) for i, w in enumerate(words)]
    print("\n- text_a:", ex.text_a[:200], "\n  text_b(target idx):", ex.text_b,
          "\n  tokens(cả câu):", len(whole), "| tổng tokens(từng từ):", sum(per_word), "| max_seq_length: 256")
    print("  tokens đầu:", whole[:12])

transformers 4.48.3
  73:         input_mask += [0] * len(padding)
  74:         segment_ids += [0] * len(padding)
  75: 
  76:         assert len(input_ids) == max_seq_length
  77:         assert len(input_mask) == max_seq_length
-----
  74:         segment_ids += [0] * len(padding)
  75: 
  76:         assert len(input_ids) == max_seq_length
  77:         assert len(input_mask) == max_seq_length
  78:         assert len(segment_ids) == max_seq_length
-----
  75: 
  76:         assert len(input_ids) == max_seq_length
  77:         assert len(input_mask) == max_seq_length
  78:         assert len(segment_ids) == max_seq_length
  79: 
-----


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

số ví dụ: 101975 | tokenizer: RobertaTokenizerFast
OK, converter chạy sạch 5,000 ví dụ đầu


## 4. Train trên toàn bộ train, chọn checkpoint theo dev

In [14]:
def cli(d): return " ".join(f"--{k} {v}" for k, v in d.items())
train_args = dict(task_name="vua", model_type=MODEL_TYPE, bert_model=BERT_MODEL, data_dir=DEVSEL,
                  do_train="True", do_eval="True", **EXTRA_ARGS)
cmd = f"cd {REPO} && {TQDM} CUDA_VISIBLE_DEVICES=0 python main.py {cli(train_args)} 2>&1 | tee {OUT}/train.log"
print(cmd)
t0 = time.time()
get_ipython().system(cmd)
TRAIN_SECONDS = time.time() - t0
print(f"train+dev-eval: {TRAIN_SECONDS/60:.1f} phút")
# `| tee` làm mất exit code -> tự kiểm tra log để không chạy tiếp các ô sau khi train đã chết
_log = open(f"{OUT}/train.log", errors="ignore").read()
if "Traceback (most recent call last)" in _log:
    raise RuntimeError("Train bị lỗi, xem 40 dòng cuối train.log:\n" + "\n".join(_log.splitlines()[-40:]))

cd /kaggle/working/MelBERT && TQDM_MININTERVAL=60 CUDA_VISIBLE_DEVICES=0 python main.py --task_name vua --model_type BERT_BASE --bert_model roberta-base --data_dir /kaggle/working/data_devsel/VUA18 --do_train True --do_eval True 2>&1 | tee /kaggle/working/results_BERT_BASE/train.log


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


device: cuda n_gpu: 1
I0000 00:00:1794871531.753392     129 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.13/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
***** Running training *****
  Batch size = 32
  Num steps = 9561
{'main_conf_path': './', 'bert_model': 'roberta-base', 'data_dir': '/kaggle/working/data_devsel/VUA18', 'task_name': 'vua', 'model_type': 'BERT_BASE', 'classifier_hidden': 768, 'lr_schedul

In [15]:
import glob
pats = ["pytorch_model.bin", "model.safetensors"]
ck = sorted([p for pat in pats for p in glob.glob(f"{REPO}/saves/**/{pat}", recursive=True)], key=os.path.getmtime)
assert ck, ("Không có checkpoint trong saves/ -> train chưa chạy xong hoặc đã lỗi. "
            "Xem ô train / train.log, đừng chạy tiếp các ô sau.")
CKPT = os.path.dirname(ck[-1]); print("checkpoint:", CKPT, os.listdir(CKPT))
shutil.copytree(CKPT, f"{OUT}/checkpoint", dirs_exist_ok=True)
shutil.copy(f"{REPO}/main_config.cfg", f"{OUT}/main_config.cfg")
!grep -n "epoch\|f1\|Best" {OUT}/train.log | tail -30

checkpoint: /kaggle/working/MelBERT/saves/roberta-base/0_20261116-2325 ['vocab.json', 'experiments.log', 'config.json', 'training_args.bin', 'merges.txt', 'main_config.cfg', 'pytorch_model.bin']
11:{'main_conf_path': './', 'bert_model': 'roberta-base', 'data_dir': '/kaggle/working/data_devsel/VUA18', 'task_name': 'vua', 'model_type': 'BERT_BASE', 'classifier_hidden': 768, 'lr_schedule': 'warmup_linear', 'warmup_epoch': 2, 'drop_ratio': 0.2, 'kfold': 10, 'num_bagging': 0, 'bagging_index': 0, 'use_pos': True, 'use_local_context': True, 'max_seq_length': 150, 'do_train': True, 'do_test': True, 'do_eval': True, 'do_lower_case': False, 'class_weight': 3, 'train_batch_size': 32, 'eval_batch_size': 8, 'learning_rate': 3e-05, 'num_train_epoch': 3, 'no_cuda': False, 'seed': 42, 'main_config': OrderedDict({'args': OrderedDict({'bert_model': 'roberta-base', 'data_dir': '/kaggle/working/data_devsel/VUA18', 'task_name': 'vua', 'model_type': 'BERT_BASE', 'classifier_hidden': 768, 'lr_schedule': 'war

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## 5. Xuất dự đoán (dev & test) bằng checkpoint đã chọn
Script nạp checkpoint bằng chính hàm của tác giả (`load_pretrained_model`, `load_trained_model`, `load_test_data`) rồi lưu từng token: nhãn thật, nhãn dự đoán, xác suất, câu, POS, w_index.

In [16]:
%%writefile /kaggle/working/export_preds.py
import sys, os, json, argparse, csv
import numpy as np, pandas as pd, torch
from tqdm import tqdm
ap = argparse.ArgumentParser()
for k in ["repo", "ckpt", "data_dir", "tsv", "out", "bert_model"]: ap.add_argument("--" + k, required=True)
a = ap.parse_args()
sys.path.insert(0, a.repo); os.chdir(a.repo)
from utils import Config, Logger
from main import load_pretrained_model, load_trained_model
from run_classifier_dataset_utils import processors, output_modes
from data_loader import load_test_data
from transformers import AutoTokenizer

tmp = a.out + "_tmp"; os.makedirs(tmp, exist_ok=True)
args = Config(main_conf_path=a.ckpt)
args.bert_model = a.bert_model          # nạp kiến trúc + tokenizer từ roberta-base, rồi nạp trọng số đã train
args.data_dir = a.data_dir; args.log_dir = a.ckpt; args.task_name = "vua"
args.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
args.n_gpu = 1 if torch.cuda.is_available() else 0
logger = Logger(tmp)
processor = processors["vua"](); label_list = processor.get_labels(); args.num_labels = len(label_list)
tokenizer = AutoTokenizer.from_pretrained(a.bert_model, do_lower_case=args.do_lower_case)
model = load_trained_model(args, load_pretrained_model(args), tokenizer)
n_params = sum(p.numel() for p in model.parameters())
all_guids, dl = load_test_data(args, logger, processor, "vua", label_list, tokenizer, output_modes["vua"])

model.eval(); rows = []
for batch in tqdm(dl, desc="predict", mininterval=30):
    batch = tuple(t.to(args.device) for t in batch)
    with torch.no_grad():
        if args.model_type in ["MELBERT_MIP", "MELBERT"]:
            ids, mask, seg, lab, idx, ids2, mask2, seg2 = batch
            lg = model(ids, ids2, target_mask=(seg == 1), target_mask_2=seg2, attention_mask_2=mask2,
                       token_type_ids=seg, attention_mask=mask)
        else:
            ids, mask, seg, lab, idx = batch
            lg = model(ids, target_mask=(seg == 1), token_type_ids=seg, attention_mask=mask)
    lg = lg.detach().cpu().numpy()
    for i, l, r in zip(idx.cpu().numpy(), lab.cpu().numpy(), lg):
        rows.append((int(i), int(l), float(r[0]), float(r[1])))
df = pd.DataFrame(rows, columns=["row", "label_loader", "lp0", "lp1"]).sort_values("row").reset_index(drop=True)
df["pred"] = (df.lp1 > df.lp0).astype(int)
df["prob_metaphor"] = 1 / (1 + np.exp(df.lp0 - df.lp1))
df["guid"] = [all_guids[i] for i in df.row]
tsv = pd.read_csv(a.tsv, sep="\t", quoting=csv.QUOTE_NONE, dtype=str, keep_default_na=False)
assert len(tsv) == len(df), f"số dòng tsv {len(tsv)} != số dự đoán {len(df)}"
assert (tsv.label.astype(int).values == df.label_loader.values).all(), "nhãn trong tsv không khớp thứ tự dự đoán"
df["label"] = df.label_loader
for c in ["index", "sentence", "POS", "w_index"]: df[c] = tsv[c].values
df.drop(columns=["label_loader", "lp0", "lp1"]).to_csv(a.out, index=False)
json.dump({"n_params": int(n_params), "n_rows": len(df)}, open(a.out + ".meta.json", "w"))
print("saved", a.out, "params", n_params)

Writing /kaggle/working/export_preds.py


In [17]:
for split, ddir, tsv in [("dev",  DEVSEL,   f"{DEVSEL}/dev.tsv"),
                         ("test", DATA_DIR, f"{DATA_DIR}/test.tsv")]:
    c = (f"cd {REPO} && CUDA_VISIBLE_DEVICES=0 {TQDM} python {WORK}/export_preds.py --repo {REPO} --ckpt {CKPT} "
         f"--bert_model {BERT_MODEL} --data_dir {ddir} --tsv {tsv} --out {OUT}/pred_{split}.csv")
    get_ipython().system(c)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


I0000 00:00:1794880694.516180     178 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
***** Running evaluation *****
predict: 100%|██████████████████████████████| 4282/4282 [05:02<00:00, 14.16it/s]
saved /kaggle/working/results_BERT_BASE/pred_dev.csv params 124649474


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


I0000 00:00:1794881036.755776     202 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
***** Running evaluation *****
predict: 100%|██████████████████████████████| 5494/5494 [06:24<00:00, 14.28it/s]
saved /kaggle/working/results_BERT_BASE/pred_test.csv params 124649474


## 6. P/R/F1 (lớp ẩn dụ = 1) — tổng thể và theo POS

In [18]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
def prf(d):
    p, r, f, _ = precision_recall_fscore_support(d.label, d.pred, average="binary", pos_label=1, zero_division=0)
    return dict(n=len(d), P=round(100*p, 1), R=round(100*r, 1), F1=round(100*f, 1), Acc=round(100*accuracy_score(d.label, d.pred), 1))
res = {}
for split in ["dev", "test"]:
    d = pd.read_csv(f"{OUT}/pred_{split}.csv", keep_default_na=False)
    res[split] = {"ALL": prf(d)}
    for pos in ["VERB", "NOUN", "ADJ", "ADV"]:
        res[split][pos] = prf(d[d.POS.str.upper() == pos])
for split in res: print(f"\n== {split} =="); print(pd.DataFrame(res[split]).T)
print("\nCông bố (README tác giả, MelBERT/RoBERTa, 1 lượt): VUA-18 F1 ≈ 78.5 ; VUA-verb F1 ≈ 75.7")
print("Số token VERB test (README: 5,873):", res["test"]["VERB"]["n"])
json.dump(res, open(f"{OUT}/metrics.json", "w"), indent=2)


== dev ==
            n     P     R    F1   Acc
ALL   34253.0  76.6  77.8  77.2  94.0
VERB   6910.0  71.4  78.8  74.9  89.8
NOUN   6858.0  77.9  69.3  73.4  93.2
ADJ    3213.0  62.3  63.4  62.8  92.0
ADV    2068.0  71.7  64.3  67.8  95.5

== test ==
            n     P     R    F1   Acc
ALL   43947.0  78.6  76.1  77.4  93.7
VERB   9247.0  73.3  75.8  74.6  89.4
NOUN   8583.0  76.1  64.0  69.5  91.6
ADJ    3965.0  68.0  61.2  64.4  90.8
ADV    3054.0  78.2  66.0  71.6  95.8

Công bố (README tác giả, MelBERT/RoBERTa, 1 lượt): VUA-18 F1 ≈ 78.5 ; VUA-verb F1 ≈ 75.7
Số token VERB test (README: 5,873): 9247


In [19]:
# (Tuỳ chọn) chạy đường test chính thức của tác giả để đối chiếu với kết quả trên.
# Nếu lỗi do cách Config đọc cờ True/False, bỏ qua — script export ở trên đã dùng chính các hàm của tác giả.
c = (f"cd {REPO} && {TQDM} CUDA_VISIBLE_DEVICES=0 python main.py --task_name vua --model_type {MODEL_TYPE} "
     f"--bert_model {CKPT} --data_dir {DATA_DIR} --do_train False --do_eval True --do_test True 2>&1 | tee {OUT}/official_test.log | tail -15")
get_ipython().system(c)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


device: cuda n_gpu: 1
I0000 00:00:1794881464.819682     227 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Some weights of RobertaModel were not initialized from the model checkpoint at /kaggle/working/MelBERT/saves/roberta-base/0_20261116-2325 and are newly initialized: ['embeddings.LayerNorm.bias', 'embeddings.LayerNorm.weight', 'embeddings.position_embeddings.weight', 'embeddings.token_type_embeddings.weight', 'embeddings.word_embeddings.weight', 'encoder.layer.0.attention.output.LayerNorm.bias', 'encoder.layer.0.attention.output.LayerNorm.weight', 'encoder.layer.0.attention.output.dense.bias', 'encoder.layer.0.attention.output.dense.weight', 'encoder.layer.0.attention.self.key.bias', 'encoder.layer.0.attention.self.key.weight', 'encoder.layer.0.attention.self.query.bias', 'encoder.layer.0.attention.self.query.weight', 'encoder.layer.0.attention.self.value.bias', 'encoder.layer.0.attention.self.value.weight', 'encoder.layer.0.intermediate.dense

## 7. Lưu thông tin chạy (thời gian, tham số, phần cứng, thay đổi so với tác giả)

In [20]:
import torch, transformers, sys
meta = json.load(open(f"{OUT}/pred_test.csv.meta.json"))
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip()
info = dict(model_type=MODEL_TYPE, bert_model=BERT_MODEL, repo_commit=COMMIT, extra_args=EXTRA_ARGS,
            train_minutes=round(TRAIN_SECONDS/60, 1), n_params=meta["n_params"], gpu=gpu,
            python=sys.version.split()[0], torch=torch.__version__, transformers=transformers.__version__,
            data_stats=stats, differences_from_authors=DIFFS, metrics=res)
json.dump(info, open(f"{OUT}/run_info.json", "w"), indent=2, ensure_ascii=False)
print(json.dumps({k: v for k, v in info.items() if k not in ("metrics", "data_stats")}, indent=2, ensure_ascii=False))
!du -sh {OUT}/*

{
  "model_type": "BERT_BASE",
  "bert_model": "roberta-base",
  "repo_commit": "11b0f78dae93c71c56b805dc28d810f6050c86c4",
  "extra_args": {},
  "train_minutes": 153.0,
  "n_params": 124649474,
  "gpu": "Tesla T4, 15360 MiB\nTesla T4, 15360 MiB",
  "python": "3.13.16",
  "torch": "2.11.0+cu130",
  "transformers": "4.48.3",
  "differences_from_authors": [
    "transformers==4.48.3 thay vì 4.2.2 (python mới trên Kaggle, torch mới thay vì 1.6)",
    "Chọn checkpoint theo dev: thư mục train có test.tsv:=dev.tsv (code gốc chọn theo tập do load_test_data đọc). Test thật chỉ dùng sau khi train xong.",
    "Chạy 1 GPU (CUDA_VISIBLE_DEVICES=0), không DataParallel.",
    "run_classifier_dataset_utils.py: segment_ids đệm thêm 1 cho [SEP] cuối (lỗi độ dài trong code gốc, assert len(segment_ids)==max_seq_length luôn fail)"
  ]
}
477M	/kaggle/working/results_BERT_BASE/checkpoint
4.0K	/kaggle/working/results_BERT_BASE/main_config.cfg
4.0K	/kaggle/working/results_BERT_BASE/metrics.json
12K	/kaggle/wo

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


**Đã lưu trong `results_<MODEL_TYPE>/`:** `train.log`, `main_config.cfg`, `checkpoint/` (best theo dev), `pred_dev.csv`, `pred_test.csv`, `metrics.json`, `run_info.json`.
Sau khi chạy xong: *Save Version → Save & Run All (Commit)* để Output được giữ lại, hoặc đẩy `checkpoint/` lên Kaggle Dataset / Drive rồi ghi link vào báo cáo.

**Cần ghi vào báo cáo:** các mục `differences_from_authors` trong `run_info.json`; nếu `main_config.cfg` khác hyperparameter trong paper thì nêu rõ.